<a href="https://colab.research.google.com/github/piramid678-arch/connect-ai/blob/main/%E1%84%89%E1%85%B5%E1%86%AF%E1%84%92%E1%85%A5%E1%86%B7_AI%E1%84%83%E1%85%AE%E1%84%80%E1%85%A2%E1%84%92%E1%85%A1%E1%86%B8%E1%84%8E%E1%85%B5%E1%84%80%E1%85%B5_model_merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 AI 두 개 합치기 — Model Merging 실험

**가중치 수술(weight surgery)의 가장 쉽고 안전한 첫 실험.**
두 AI 모델의 *가중치를 수학적으로 섞어서* 두 능력을 다 가진 모델 하나를 만든다.
학습(GPU 오래 돌리기) 없이, 몇 분이면 끝난다.

> 💡 원리: AI의 능력은 가중치 공간의 '방향(벡터)'이다. 두 모델의 가중치를 보간(SLERP)하면 두 능력이 한 모델에 섞인다.

**준비**: 위 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 (무료)

## 1단계 — 도구 설치 (mergekit)

In [22]:
!pip install -q torch transformers accelerate
!git clone -q https://github.com/arcee-ai/mergekit.git
%cd mergekit
!pip install -q -e .
print('✅ mergekit 설치 완료')

fatal: destination path 'mergekit' already exists and is not an empty directory.
/content/mergekit/mergekit
ERROR: file:///content/mergekit/mergekit does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
✅ mergekit 설치 완료


In [26]:
!pip install -U -q "protobuf<5.0.0" "transformers>=4.44.0"
import google.protobuf
print(f"✅ Protobuf 버전 수정 완료: {google.protobuf.__version__}")

✅ Protobuf 버전 수정 완료: 4.25.9


In [27]:
!pip install -U -q --force-reinstall "protobuf<5.0.0" "transformers>=4.44.0" "peft" "accelerate"
import google.protobuf
import transformers
print(f"✅ Protobuf 버전: {google.protobuf.__version__}")
print(f"✅ Transformers 버전: {transformers.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [30]:
!pip uninstall -y protobuf transformers
!pip install -q "protobuf<5.0.0" "transformers>=4.44.0" "mergekit" "peft" "accelerate"
import google.protobuf
import mergekit
print(f"✅ Protobuf 버전: {google.protobuf.__version__}")
print("✅ Mergekit 라이브러리 로드 성공")

Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.10.6 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatibl

In [33]:
!pip uninstall -y protobuf transformers
!pip install -q "protobuf<5.0.0" "transformers>=4.44.0" "mergekit" "peft" "accelerate"
import google.protobuf
import mergekit
print(f"✅ Protobuf 버전: {google.protobuf.__version__}")
print("✅ Mergekit 라이브러리 로드 성공")

Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.10.6 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 15.0.0

In [ ]:
import os
import sys
import importlib

# 1. 의존성 꼬임을 해결하기 위해 핵심 라이브러리들을 삭제 후 특정 버전으로 재설치
!pip uninstall -y protobuf transformers tensorflow torch torchvision
!pip install -q "protobuf<5.0.0" "transformers==4.44.2" "torch==2.4.1" "torchvision==0.19.1" "mergekit" "peft" "accelerate"

# 2. Python 세션에 변경사항 강제 반영
if 'google.protobuf' in sys.modules:
    importlib.reload(sys.modules['google.protobuf'])

import google.protobuf
import transformers
import torch

print(f"✅ Protobuf 버전: {google.protobuf.__version__}")
print(f"✅ Transformers 버전: {transformers.__version__}")
print(f"✅ Torch 버전: {torch.__version__}")
print("✅ 환경 복구가 완료되었습니다. 이제 아래의 병합 셀을 다시 실행하세요.")

### 🚀 Step 3 (Retry): Model Merging
Now that the environment is stabilized, let's run the merge command again.

In [ ]:
import os
# Ensure we are in the correct directory
%cd /content

# Run the merge command
!mergekit-yaml merge_config.yaml ./merged_model --cuda --allow-crimes

if os.path.exists('./merged_model'):
    print('✅ Success! Model merged at ./merged_model')
else:
    print('❌ Merge failed. Please check the error logs above.')

### 🚀 Step 3: 모델 병합 실행
라이브러리 충돌 문제가 해결되었으므로, 이제 병합을 시작합니다.

In [36]:
import os
# 작업 디렉토리를 /content로 고정
%cd /content

# 병합 명령 실행 (이전 오류 해결 후 재시도)
!mergekit-yaml merge_config.yaml ./merged_model --cuda --allow-crimes

if os.path.exists('./merged_model'):
    print('✅ 성공! 모델이 ./merged_model 폴더에 생성되었습니다.')
else:
    print('❌ 병합에 실패했습니다. 위의 로그를 확인해주세요.')

/content
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2262, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2496, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2494, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<froze

### 🚀 위 셀 완료 후 아래의 3단계를 다시 실행하세요.
환경 재설치가 완료되었다면 이제 `6QZ1a_LXUM2m` 셀이 정상 작동할 것입니다.

### 🚀 위 셀 완료 후 아래의 3단계를 다시 실행하세요.
환경 재설치가 완료되었다면 이제 `6QZ1a_LXUM2m` 셀이 정상 작동할 것입니다.

### 🚀 위 셀 완료 후 아래의 3단계를 다시 실행하세요.
라이브러리 재설치가 완료되었다면 이제 `6QZ1a_LXUM2m` 셀이 정상 작동할 것입니다.

### 🛠️ 환경 수정 후 다시 시도
위 셀을 실행한 후, 아래의 3단계(`6QZ1a_LXUM2m`) 셀을 다시 실행해 주세요.

## 2단계 — 합칠 두 모델 + 방법 정하기

⚠️ **두 모델은 반드시 같은 베이스(같은 아키텍처·크기)여야** 합쳐진다.
아래는 작은 Qwen2.5-1.5B 계열 예시 (무료 T4에서 안전). 원하는 같은-베이스 모델로 바꿔도 된다.

- `MODEL_A`, `MODEL_B`: 합칠 두 모델 (HuggingFace 이름)
- `merge_method: slerp`: 두 가중치를 매끄럽게 보간 (가장 무난한 방법)
- `t: 0.5`: 0.5 = 반반. 0에 가까우면 A 쪽, 1에 가까우면 B 쪽.

In [31]:
config = '''
slices:
  - sources:
      - model: Qwen/Qwen2.5-1.5B-Instruct
        layer_range: [0, 28]
      - model: Qwen/Qwen2.5-Coder-1.5B-Instruct
        layer_range: [0, 28]
merge_method: slerp
base_model: Qwen/Qwen2.5-1.5B-Instruct
parameters:
  t: 0.5
dtype: bfloat16
'''
with open('merge_config.yaml', 'w') as f:
    f.write(config)
print('✅ 설정 저장 — 일반 Qwen(대화) + Coder Qwen(코딩)을 반반 섞는다')

✅ 설정 저장 — 일반 Qwen(대화) + Coder Qwen(코딩)을 반반 섞는다


## 3단계 — 합치기 실행 (수술!) 🔪

두 모델을 받아서 가중치를 섞는다. 처음엔 모델 다운로드 때문에 몇 분 걸린다.

In [34]:
!mergekit-yaml merge_config.yaml ./merged_model --cuda --allow-crimes
print('✅ 합치기 완료 → ./merged_model 에 새 모델 탄생')

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2262, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2496, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2494, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importl

## 4단계 — 합쳐진 모델 테스트 🧪

대화도 되고 코딩도 되는지 — 두 능력이 한 모델에 들어왔는지 확인.

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

model_path = '/content/mergekit/merged_model'

if os.path.exists(model_path):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map='auto')

    def ask(q):
        msgs = [{'role':'user','content':q}]
        inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
        out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
        print(tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

    print('=== 대화 능력 ==='); ask('1인 기업가에게 좋은 아침 루틴 3가지 추천해줘')
    print('\n=== 코딩 능력 ==='); ask('파이썬으로 피보나치 수열 함수 짧게 써줘')
else:
    print(f'❌ 모델 폴더를 찾을 수 없습니다: {model_path}\n3단계가 성공적으로 완료되었는지 확인해주세요.')

❌ 모델 폴더를 찾을 수 없습니다: /content/mergekit/merged_model
3단계가 성공적으로 완료되었는지 확인해주세요.


In [14]:
import os
# 실행 경로를 /content로 변경
%cd /content

# mergekit 실행 (설치된 라이브러리 사용)
!mergekit-yaml merge_config.yaml ./merged_model --cuda --allow-crimes

if os.path.exists('./merged_model'):
    print('✅ 합치기 완료 → ./merged_model 에 새 모델 탄생')
else:
    print('❌ 모델 합치기에 실패했습니다. 로그를 확인해주세요.')

2026-06-15 05:44:34.910284: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/mergekit/mergekit/options.py", line 15, in <module>
    from mergekit.common import parse_kmb
  File "/content/mergekit/mergekit/common.py", line 24, in <module>
    import peft
  File "/usr/local/lib/python3.12/dist-packages/peft/__init__.py", line 17, in <module>
    from .auto import (
  File "/usr/local/lib/python3.12/dist-packages/peft/auto.py", line 31, in <module>
    from .config import PeftConfig
  File "/usr/local/lib/python3.12/dist-packages/peft/config.py", line 30, in <module>
    from .utils import CONFI

## 5단계 (선택) — 우리 앱(Connect AI)에서 돌리기 🖥️

합친 모델을 GGUF로 변환하면 Connect AI 데스크톱 앱에서 바로 실행할 수 있다.
(llama.cpp 변환 → 양자화 → 앱의 🤖 내 AI 에서 불러오기)

In [15]:
%cd /content
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
# FP16 GGUF로 변환
!python llama.cpp/convert_hf_to_gguf.py /content/mergekit/merged_model --outfile my-merged.gguf --outtype f16
print('✅ my-merged.gguf 생성 — 다운로드해서 Connect AI 앱 모델 폴더에 넣으면 끝')

/content
fatal: destination path 'llama.cpp' already exists and is not an empty directory.
ERROR:hf-to-gguf:Error: /content/mergekit/merged_model is not a directory
✅ my-merged.gguf 생성 — 다운로드해서 Connect AI 앱 모델 폴더에 넣으면 끝


In [16]:
from google.colab import files

# 파일이 생성되었는지 확인 후 다운로드 실행
if os.path.exists('/content/my-merged.gguf'):
    files.download('/content/my-merged.gguf')
else:
    print('❌ 다운로드할 파일을 찾을 수 없습니다. 5단계의 변환 과정을 먼저 완료해주세요.')

❌ 다운로드할 파일을 찾을 수 없습니다. 5단계의 변환 과정을 먼저 완료해주세요.


---
## 🎓 무슨 일이 일어난 건가

- 두 모델의 **가중치를 SLERP(구면 선형 보간)로 섞었다** — 학습이 아니라 수학적 합성
- 결과: "대화 잘하는 Qwen" + "코딩 잘하는 Qwen" = **둘 다 하는 모델**
- 이게 **Model Soups / TIES / DARE** 계열 연구(weight 산술)의 기초

**다음 실험 아이디어:**
- `t` 값을 0.3, 0.7로 바꿔서 어느 능력이 세지는지 비교
- 한국어 모델 + 코딩 모델 합치기
- `merge_method`를 `ties`나 `dare_ties`로 바꿔보기 (3개 이상도 합쳐짐)

> 🔬 이건 '검열 제거(abliteration)'와 다른, 안전하고 건설적인 weight 수술이다. 능력을 **더하는** 방향.